## PART ONE - Maximising a function.

Objective: Find the maximum value of f(x,y,z) , where

f(x,y,z) = −e−(x−0.55z)2cos 81(x+z+0.12y) +sin44(y+0.03(x−z))
−cos 69sin(xz+0.07y) −sin1.32(x−z) −0.32(z+0.22xz)2esin(63(z−x))
+cos(77x)+sin(71z)−e−y2cos(75y)+sin(43y)+cos(73y)+0.05(x2+y2+z2).

Constraints: Thesolution must be subject to the (hard) constraints:

0≤x≤5
0≤y≤5
0≤z≤5



In [19]:
import random
import math

# function to maximise
def f(x, y, z):
    return (
        -math.exp(-(x - 0.55*z)**2) * math.cos(81*(x + z + 0.12*y))
        + math.sin(44*(y + 0.03*(x - z)))
        - math.cos(69 * math.sin(x*z + 0.07*y))
        - math.sin(1.32*(x - z))
        - 0.32*(z + 0.22*x*z)**2 * math.exp(math.sin(63*(z - x)))
        + math.cos(77*x)
        + math.sin(71*z)
        - math.exp(-y**2) * math.cos(75*y)
        + math.sin(43*y)
        + math.cos(73*y)
        + 0.05*(x**2 + y**2 + z**2)
    )

# keeps values between 0 and 5
def clip(value):
    if value < 0:
        return 0
    elif value > 5:
        return 5
    else:
        return value

best_f = -999999
best_x = 0
best_y = 0
best_z = 0

repeats = 50      # how many random starting points
steps = 5000      # how many hill-climb steps per start
sigma = 0.2       # step size

for j in range(repeats):
    # random starting point
    x = random.uniform(0, 5)
    y = random.uniform(0, 5)
    z = random.uniform(0, 5)

    current_f = f(x, y, z)

    for i in range(steps):
        # make a small random move
        x_new = clip(x + random.gauss(0, sigma))
        y_new = clip(y + random.gauss(0, sigma))
        z_new = clip(z + random.gauss(0, sigma))

        new_f = f(x_new, y_new, z_new)

        # keep new point only if it is better
        if new_f > current_f:
            x = x_new
            y = y_new
            z = z_new
            current_f = new_f

    print("Run", j+1, ": x =", x, "y =", y, "z =", z, "f =", current_f)

    # check if this run is the best overall
    if current_f > best_f:
        best_f = current_f
        best_x = x
        best_y = y
        best_z = z

print("\nBest answer found:")
print("x =", best_x)
print("y =", best_y)
print("z =", best_z)
print("f(x,y,z) =", best_f)

Run 1 : x = 4.244584603332542 y = 3.090389032729147 z = 0.8131362015070647 f = 7.252394707503744
Run 2 : x = 1.7804600190007254 y = 2.4977728403834365 z = 2.849667822946375 f = 5.732047108101227
Run 3 : x = 3.1768722414147237 y = 2.672708965188477 z = 0.10719782487828006 f = 7.166037913512944
Run 4 : x = 0.402387978801952 y = 0.3466198861058106 z = 0.9941843431172531 f = 6.43438682386646
Run 5 : x = 3.5142145310823985 y = 2.250765404892984 z = 1.7029807380578261 f = 3.8919199254815453
Run 6 : x = 1.8764151382589336 y = 5 z = 2.0520658350240613 f = 5.784222757848062
Run 7 : x = 1.6349567623097168 y = 3.3754818779728524 z = 2.4165331030668797 f = 5.124113842787869
Run 8 : x = 4.082812915770039 y = 3.690266705190726 z = 1.168765285669654 f = 6.477180156046398
Run 9 : x = 0.4093274770107981 y = 3.1070288397809165 z = 1.9690231207760602 f = 6.452148642660081
Run 10 : x = 2.0347693094279737 y = 1.2009020367467562 z = 3.2071040908905863 f = 5.923738699159162
Run 11 : x = 2.1179732935769304 y 

For the above, I used a hill-climbing search method to find the best values of three variables. The algorithm starts from a random point and then makes small random changes to the variables. If a new solution improves the result then it is accepted, otherwise, it is rejected. This process is then repeated many times to then gradually improve the solution. Since this method can get stuck in a suboptimal solution, I repeated the search from multiple random searching points and then selected the best overall result. Any values outside the allowed range were adjusted to ensure the hard constraints were satisfied. 

## PART TWO - Distribution Network
A major supermarket is updating its delivery network. They have 2 main warehouses
(W1 and W2) and 23 stores at locations (1-23). Each day they must carry out a daily
delivery from their two warehouses to all 23 stores, with the vehicles returning to the ware
houses at the end of the day. The location co-ordinates and distance matrix is in a seperate datafile. 

Given the aim is to minimise the total daily costs, find the best strategy such that
every store receives its delivery and the warehouses have the correct number of vehicles at
the end of the day to carry out the deliveries the following day.
Questions: Which stores should each warehouse supply? How many vans or lorries does
each warehouse require? What routes should each vehicle take? What is the total cost?

In [2]:
!pip install openpyxl

In [5]:
import pandas as pd
import math
import itertools

# Load data.
file_name = "Part 2 Data (2026).xlsx"
df = pd.read_excel(file_name)
df = df.iloc[:, 0:3]
df.columns = ["location", "x", "y"]
df = df.dropna()

coords = {}
for _, row in df.iterrows():
    coords[str(row["location"]).strip()] = (float(row["x"]), float(row["y"]))

coords["w1"] = coords["W1"]
coords["w2"] = coords["W2"]

stores = [str(i) for i in range(1, 24)]
warehouses = ["w1", "w2"]

# Distance functions.
def dist(a, b):
    x1, y1 = coords[a]
    x2, y2 = coords[b]
    return math.hypot(x1 - x2, y1 - y2)

def route_dist(w, r):
    if not r:
        return 0.0
    d = dist(w, r[0])
    for i in range(len(r) - 1):
        d += dist(r[i], r[i + 1])
    d += dist(r[-1], w)
    return d

# Route construction.
def nn_route(w, s):
    s = s[:]
    cur = w
    r = []
    while s:
        nxt = min(s, key=lambda x: dist(cur, x))
        r.append(nxt)
        s.remove(nxt)
        cur = nxt
    return r

def two_opt(w, r):
    best = r[:]
    improved = True
    while improved:
        improved = False
        best_d = route_dist(w, best)
        for i in range(len(best) - 1):
            for j in range(i + 1, len(best)):
                cand = best[:]
                cand[i:j+1] = reversed(cand[i:j+1])
                d = route_dist(w, cand)
                if d < best_d - 1e-9:
                    best = cand
                    best_d = d
                    improved = True
    return best

def best_route(w, g):
    if len(g) <= 8:
        best = None
        best_d = float("inf")
        for p in itertools.permutations(g):
            d = route_dist(w, list(p))
            if d < best_d:
                best = list(p)
                best_d = d
        return best, best_d
    r = nn_route(w, g)
    r = two_opt(w, r)
    return r, route_dist(w, r)

# Cost model.
def route_cost(w, g):
    if not g:
        return None, 0.0, 0.0
    r, m = best_route(w, g)
    if len(g) <= 5:
        return r, m, 2 * m
    return r, m, 3 * m

# Sweep ordering.
def angle(w, s):
    wx, wy = coords[w]
    sx, sy = coords[s]
    return math.atan2(sy - wy, sx - wx)

def order_angle(w, s):
    return sorted(s, key=lambda x: angle(w, x))

# Partition routes with dp.
def best_partition(w, s):
    if not s:
        return [], 0.0, 0.0

    base = order_angle(w, s)
    best_cost = float("inf")
    best_sol = None
    best_miles = 0.0

    n = len(base)
    variants = []
    for i in range(n):
        rot = base[i:] + base[:i]
        variants += [rot, list(reversed(rot))]

    for seq in variants:
        m = len(seq)
        dp = [float("inf")] * (m + 1)
        prev = [None] * (m + 1)
        dp[0] = 0.0

        for i in range(m):
            if dp[i] == float("inf"):
                continue
            for k in range(1, min(18, m - i) + 1):
                g = seq[i:i+k]
                r, miles, cost = route_cost(w, g)
                if dp[i] + cost < dp[i+k]:
                    dp[i+k] = dp[i] + cost
                    prev[i+k] = (i, r, miles, cost)

        routes = []
        miles = 0.0
        i = m
        while i > 0:
            j, r, m_i, c = prev[i]
            routes.append((r, m_i, c))
            miles += m_i
            i = j

        routes.reverse()

        if dp[m] < best_cost:
            best_cost = dp[m]
            best_sol = routes
            best_miles = miles

    return best_sol, best_miles, best_cost

# Evaluate full solution.
def evaluate(assign):
    all_routes = []
    total_miles = 0.0
    total_cost = 0.0

    for w in warehouses:
        sol, m, c = best_partition(w, assign[w])
        for r, mi, co in sol:
            all_routes.append((w, r, mi, co))
        total_miles += m
        total_cost += c

    return all_routes, total_miles, total_cost

# Initial assignment.
def init_assign():
    a = {"w1": [], "w2": []}
    for s in stores:
        if dist("w1", s) <= dist("w2", s):
            a["w1"].append(s)
        else:
            a["w2"].append(s)
    return a

# Local search improvement.
def improve(a):
    cur = {"w1": a["w1"][:], "w2": a["w2"][:]}
    _, _, best_cost = evaluate(cur)

    improved = True
    while improved:
        improved = False
        for w in warehouses:
            other = "w2" if w == "w1" else "w1"
            for s in cur[w]:
                trial = {"w1": cur["w1"][:], "w2": cur["w2"][:]}
                trial[w].remove(s)
                trial[other].append(s)

                _, _, c = evaluate(trial)
                if c < best_cost:
                    cur = trial
                    best_cost = c
                    improved = True
                    break
            if improved:
                break

    return cur

# Solve.
assign = init_assign()
assign = improve(assign)

routes, miles, cost = evaluate(assign)

# Output.
print("Best heuristic solution found.")
print()

print("Stores supplied by W1:", sorted(assign["w1"], key=int))
print("Stores supplied by W2:", sorted(assign["w2"], key=int))
print()

w1_v = w1_l = w2_v = w2_l = 0

for w, r, m, c in routes:
    if w == "w1":
        if len(r) <= 5:
            w1_v += 1
        else:
            w1_l += 1
    else:
        if len(r) <= 5:
            w2_v += 1
        else:
            w2_l += 1

print("Vehicles required.")
print(f"W1: {w1_v} vans, {w1_l} lorries.")
print(f"W2: {w2_v} vans, {w2_l} lorries.")
print()

print("Routes.")
for i, (w, r, m, c) in enumerate(routes, 1):
    path = " -> ".join(r)
    print(f"{i}. {w.upper()} -> {path} -> {w.upper()}")
    print(f"   Stores: {len(r)}.")
    print(f"   Miles: {m:.2f}.")
    print(f"   Cost: £{c:.2f}.")
    print()

print(f"Total miles: {miles:.2f}.")
print(f"Total cost: £{cost:.2f}.")

Best heuristic solution found.

Stores supplied by W1: ['1', '6', '8', '9', '11', '12', '14', '17', '19', '20']
Stores supplied by W2: ['2', '3', '4', '5', '7', '10', '13', '15', '16', '18', '21', '22', '23']

Vehicles required.
W1: 2 vans, 0 lorries.
W2: 3 vans, 0 lorries.

Routes.
1. W1 -> 12 -> 19 -> 11 -> 9 -> 6 -> W1
   Stores: 5.
   Miles: 130.97.
   Cost: £261.94.

2. W1 -> 17 -> 8 -> 1 -> 20 -> 14 -> W1
   Stores: 5.
   Miles: 92.54.
   Cost: £185.08.

3. W2 -> 15 -> 13 -> 18 -> 7 -> W2
   Stores: 4.
   Miles: 77.50.
   Cost: £155.00.

4. W2 -> 5 -> 2 -> 4 -> 10 -> 22 -> W2
   Stores: 5.
   Miles: 152.35.
   Cost: £304.70.

5. W2 -> 16 -> 3 -> 21 -> 23 -> W2
   Stores: 4.
   Miles: 92.96.
   Cost: £185.93.

Total miles: 546.32.
Total cost: £1092.65.


The algorithm produced a feasible low-cost solution with total daily cost £1092.65. The best solution found by the heuristic used 5 vans and 0 lorries. Although this does not prove global optimality, the warehouse allocations and routes were geographically sensible, suggesting the solution is likely to be close to optimal.

## PART THREE - Real Life Example 

Create your own example of a real-life optimisation problem. This should be something
from your personal life, or something you have an interest in. It can be from any field. It
may involve using existing data, or simply approximating behaviour with simulated data
and your own model. However, it must be your own example that you create.


In [7]:
# Checking the data file is accessible. 

import pandas as pd

file_path = r"C:\Users\jess\Jupyter Notebooks\Toronto.xlsx"
df = pd.read_excel(file_path)

df.head()

,place,date,area,type,time_spent,score,must_visit,earliest_time,latest_time
0,hotel,2026-09-19,downtown,start,0.0,0,yes,08:00:00,23:00:00
1,cn tower,2026-09-19,downtown,attraction,2.0,10,yes,10:00:00,18:00:00
2,fast food lunch,2026-09-19,downtown,food,1.0,7,yes,12:00:00,14:00:00
3,st lawrence market,2026-09-19,downtown east,attraction,1.5,8,no,18:00:00,22:00:00
4,kensington market,2026-09-19,downtown west,attraction,2.0,9,no,09:00:00,17:00:00


In [6]:
import random
from copy import deepcopy
from pathlib import Path

import pandas as pd


# Set the random seed so resultds can be reproduced.
random.seed(42)

df = pd.read_excel(r"C:\Users\jess\Jupyter Notebooks\Toronto.xlsx")

# Clean the collumn names
df.columns = [col.strip().lower() for col in df.columns]

# Convert the main datatypes
df["date"] = pd.to_datetime(df["date"]).dt.date
df["time_spent"] = pd.to_numeric(df["time_spent"], errors="coerce").fillna(0.0)
df["score"] = pd.to_numeric(df["score"], errors="coerce").fillna(0.0)
df["must_visit"] = df["must_visit"].astype(str).str.strip().str.lower().eq("yes")

# Convert time values to minutes after midnight
def time_to_minutes(value):
    if pd.isna(value):
        return None
    if hasattr(value, "hour") and hasattr(value, "minute"):
        return value.hour * 60 + value.minute
    parsed = pd.to_datetime(str(value), format="%H:%M:%S", errors="coerce")
    if pd.isna(parsed):
        parsed = pd.to_datetime(str(value), errors="coerce")
    return parsed.hour * 60 + parsed.minute

df["earliest_min"] = df["earliest_time"].apply(time_to_minutes)
df["latest_min"] = df["latest_time"].apply(time_to_minutes)

# Clean small text issues in the data (I made a spelling mistake)
df["area"] = df["area"].astype(str).str.strip().str.lower()
df["type"] = df["type"].astype(str).str.strip().str.lower()
df["type"] = df["type"].replace({"attaction": "attraction"})

# Create approximate travel times between areas in minutes
AREA_TRAVEL = {
    ("downtown", "downtown"): 15,
    ("downtown", "downtown east"): 20,
    ("downtown", "downtown west"): 20,
    ("downtown", "waterfront"): 20,
    ("downtown", "midtown"): 30,
    ("downtown", "north toronto"): 50,
    ("downtown", "niagara"): 90,
    ("downtown east", "downtown east"): 15,
    ("downtown east", "downtown west"): 30,
    ("downtown east", "waterfront"): 20,
    ("downtown east", "midtown"): 35,
    ("downtown east", "north toronto"): 55,
    ("downtown east", "niagara"): 95,
    ("downtown west", "downtown west"): 15,
    ("downtown west", "waterfront"): 25,
    ("downtown west", "midtown"): 25,
    ("downtown west", "north toronto"): 45,
    ("downtown west", "niagara"): 95,
    ("waterfront", "waterfront"): 15,
    ("waterfront", "midtown"): 35,
    ("waterfront", "north toronto"): 55,
    ("waterfront", "niagara"): 95,
    ("midtown", "midtown"): 15,
    ("midtown", "north toronto"): 25,
    ("midtown", "niagara"): 100,
    ("north toronto", "north toronto"): 20,
    ("north toronto", "niagara"): 110,
    ("niagara", "niagara"): 20,
}

# Make the travel matrix symmetric
for (area_a, area_b), value in list(AREA_TRAVEL.items()):
    AREA_TRAVEL[(area_b, area_a)] = value

# Return a travel time between two areas
def travel_minutes(area_a, area_b):
    if area_a == area_b:
        return 15
    return AREA_TRAVEL.get((area_a, area_b), 40)

# Convert minutes back to a clock string
def minutes_to_clock(total_minutes):
    hours = int(total_minutes) // 60
    minutes = int(total_minutes) % 60
    return f"{hours:02d}:{minutes:02d}"

# Build a feasible schedule from a chosen visit order
def build_schedule(day_df, chosen_order):
    start_rows = day_df[day_df["type"] == "start"]

    if len(start_rows) > 0:
        start_row = start_rows.iloc[0]
        current_time = int(start_row["earliest_min"])
        current_area = start_row["area"]
        schedule_rows = [{
            "place": start_row["place"],
            "type": start_row["type"],
            "area": start_row["area"],
            "arrival": current_time,
            "start": current_time,
            "finish": current_time,
            "travel_before": 0,
            "score": 0.0,
            "must_visit": True
        }]
        day_end_limit = int(start_row["latest_min"])
    else:
        current_time = max(8 * 60, int(day_df["earliest_min"].min()))
        current_area = chosen_order[0]["area"] if chosen_order else "downtown"
        schedule_rows = []
        day_end_limit = max(23 * 60, int(day_df["latest_min"].max()))

    total_score = 0.0
    total_travel = 0
    total_wait = 0

    for row in chosen_order:
        travel_time = travel_minutes(current_area, row["area"])
        arrival_time = current_time + travel_time
        start_time = max(arrival_time, int(row["earliest_min"]))
        finish_time = start_time + int(round(float(row["time_spent"]) * 60))

        if finish_time > int(row["latest_min"]):
            return None

        schedule_rows.append({
            "place": row["place"],
            "type": row["type"],
            "area": row["area"],
            "arrival": arrival_time,
            "start": start_time,
            "finish": finish_time,
            "travel_before": travel_time,
            "score": float(row["score"]),
            "must_visit": bool(row["must_visit"])
        })

        total_score += float(row["score"])
        total_travel += travel_time
        total_wait += max(0, start_time - arrival_time)
        current_time = finish_time
        current_area = row["area"]

    if current_time > day_end_limit:
        return None

    objective_value = total_score - 0.05 * total_travel - 0.02 * total_wait

    return {
        "schedule": schedule_rows,
        "objective": objective_value,
        "raw_score": total_score,
        "travel_minutes": total_travel,
        "wait_minutes": total_wait,
        "finish_time": current_time
    }

# Create a random initial solution that always includes must visit rows
def random_solution(day_df):
    visit_rows = day_df[day_df["type"] != "start"].to_dict("records")
    must_rows = [row for row in visit_rows if row["must_visit"]]
    optional_rows = [row for row in visit_rows if not row["must_visit"]]

    chosen = must_rows[:]
    for row in optional_rows:
        if random.random() < 0.5:
            chosen.append(row)

    random.shuffle(chosen)
    return chosen

# Mutate a solution using swaps moves adds and removes
def mutate_solution(current_solution, day_df):
    candidate = deepcopy(current_solution)
    all_rows = day_df[day_df["type"] != "start"].to_dict("records")

    must_names = {
        row["place"]
        for row in all_rows
        if row["must_visit"]
    }

    chosen_names = [row["place"] for row in candidate]

    unchosen_optional = [
        row for row in all_rows
        if (row["place"] not in chosen_names) and (row["place"] not in must_names)
    ]

    chosen_optional_positions = [
        index for index, row in enumerate(candidate)
        if row["place"] not in must_names
    ]

    mutation_type = random.choice(["swap", "move", "add", "remove"])

    if mutation_type == "swap" and len(candidate) >= 2:
        i, j = random.sample(range(len(candidate)), 2)
        candidate[i], candidate[j] = candidate[j], candidate[i]

    elif mutation_type == "move" and len(candidate) >= 2:
        i, j = random.sample(range(len(candidate)), 2)
        row = candidate.pop(i)
        candidate.insert(j, row)

    elif mutation_type == "add" and len(unchosen_optional) > 0:
        row = random.choice(unchosen_optional)
        insert_at = random.randint(0, len(candidate))
        candidate.insert(insert_at, row)

    elif mutation_type == "remove" and len(chosen_optional_positions) > 0:
        remove_at = random.choice(chosen_optional_positions)
        candidate.pop(remove_at)

    return candidate

# Run a multi start hill climb for one day
def optimise_one_day(day_df, starts=250, steps_per_start=500):
    best_result = None
    best_solution = None

    for _ in range(starts):
        current_solution = random_solution(day_df)
        current_result = build_schedule(day_df, current_solution)

        if current_result is None:
            continue

        for _ in range(steps_per_start):
            new_solution = mutate_solution(current_solution, day_df)
            new_result = build_schedule(day_df, new_solution)

            if new_result is not None and new_result["objective"] > current_result["objective"]:
                current_solution = new_solution
                current_result = new_result

        if best_result is None or current_result["objective"] > best_result["objective"]:
            best_result = current_result
            best_solution = deepcopy(current_solution)

    return best_solution, best_result

# Solve the trip day by day
best_solutions = {}
best_results = {}

for trip_date in sorted(df["date"].unique()):
    day_df = df[df["date"] == trip_date].copy()
    solution, result = optimise_one_day(day_df, starts=250, steps_per_start=500)
    best_solutions[trip_date] = solution
    best_results[trip_date] = result

# Print the schedule for each day
for trip_date in sorted(best_results.keys()):
    result = best_results[trip_date]

    print("\n" + "=" * 70)
    print(f"DATE: {trip_date}")
    print("=" * 70)

    if result is None:
        print("No feasible schedule found for this date.")
        continue

    print(f"Objective Value: {result['objective']:.2f}")
    print(f"Raw Score: {result['raw_score']:.2f}")
    print(f"Travel Minutes: {result['travel_minutes']}")
    print(f"Waiting Minutes: {result['wait_minutes']}")
    print(f"Finish Time: {minutes_to_clock(result['finish_time'])}")
    print()

    schedule_df = pd.DataFrame(result["schedule"]).copy()
    schedule_df["arrival"] = schedule_df["arrival"].apply(minutes_to_clock)
    schedule_df["start"] = schedule_df["start"].apply(minutes_to_clock)
    schedule_df["finish"] = schedule_df["finish"].apply(minutes_to_clock)

    display_columns = [
        "place",
        "type",
        "area",
        "arrival",
        "start",
        "finish",
        "travel_before",
        "score",
        "must_visit"
    ]

    print(schedule_df[display_columns].to_string(index=False))

# Build one combined table for the whole trip
all_schedule_tables = []

for trip_date in sorted(best_results.keys()):
    result = best_results[trip_date]
    if result is None:
        continue

    temp_df = pd.DataFrame(result["schedule"]).copy()
    temp_df.insert(0, "date", trip_date)
    all_schedule_tables.append(temp_df)

if len(all_schedule_tables) > 0:
    final_schedule_df = pd.concat(all_schedule_tables, ignore_index=True)
    final_schedule_df["arrival"] = final_schedule_df["arrival"].apply(minutes_to_clock)
    final_schedule_df["start"] = final_schedule_df["start"].apply(minutes_to_clock)
    final_schedule_df["finish"] = final_schedule_df["finish"].apply(minutes_to_clock)

    print("\n" + "=" * 70)
    print("FULL TRIP SCHEDULE")
    print("=" * 70)
    print(final_schedule_df.to_string(index=False))


DATE: 2026-09-19
Objective Value: 41.70
Raw Score: 48.00
Travel Minutes: 110
Waiting Minutes: 40
Finish Time: 19:30

             place       type          area arrival start finish  travel_before  score  must_visit
             hotel      start      downtown   08:00 08:00  08:00              0    0.0        True
     trillium park attraction    waterfront   08:20 09:00  10:00             20    6.0       False
       berczy park attraction downtown east   10:20 10:20  11:50             20    7.0       False
   fast food lunch       food      downtown   12:10 12:10  13:10             20    7.0        True
   tono restaurant       food      downtown   13:25 13:25  15:25             15   10.0        True
          cn tower attraction      downtown   15:40 15:40  17:40             15   10.0        True
st lawrence market attraction downtown east   18:00 18:00  19:30             20    8.0       False

DATE: 2026-09-20
Objective Value: 21.90
Raw Score: 30.00
Travel Minutes: 120
Waiting Minu